# TargetRecon — Python API Demo

This notebook walks through the full TargetRecon Python API:
- Fetching a target report
- Exploring UniProt, PDB, AlphaFold, ChEMBL, and STRING-DB data
- Filtering and ranking ligands
- Exporting reports (HTML, JSON, SDF)
- Batch processing multiple targets

**Install:** `pip install targetrecon`

## 1. Installation check

In [1]:
import targetrecon
print("TargetRecon version:", targetrecon.__version__)

TargetRecon version: 0.1.12


## 2. Run a recon — single target

`targetrecon.recon()` accepts a **gene name**, **UniProt accession**, or **ChEMBL target ID**.

It fetches data from 4 sources in parallel (UniProt, PDB, AlphaFold, ChEMBL, STRING-DB) and returns a `TargetReport` object.

In [2]:
report = targetrecon.recon("EGFR")
print(f"Query resolved to: {report.uniprot.uniprot_id} / {report.uniprot.gene_name}")
print(f"Protein name    : {report.uniprot.protein_name}")
print(f"PDB structures  : {report.num_pdb_structures}")
print(f"Bioactivities   : {report.num_bioactivities}")
print(f"Unique ligands  : {report.num_unique_ligands}")

Resolving identifiers for 'EGFR'...

UniProt: P00533  |  ChEMBL: CHEMBL203

Fetching data from 4 sources in parallel...

Done!  50 structures · 1000 bioactivities · 550 unique ligands

Query resolved to: P00533 / EGFR
Protein name    : Epidermal growth factor receptor
PDB structures  : 50
Bioactivities   : 1000
Unique ligands  : 550


### Works with UniProt accessions and ChEMBL IDs too

In [3]:
# By UniProt accession
report_up = targetrecon.recon("P00533")
print("UniProt input →", report_up.uniprot.gene_name)

# By ChEMBL target ID
report_ch = targetrecon.recon("CHEMBL203")
print("ChEMBL input  →", report_ch.uniprot.gene_name)

Resolving identifiers for 'P00533'...

UniProt: P00533  |  ChEMBL: CHEMBL203

Fetching data from 4 sources in parallel...

Done!  50 structures · 1000 bioactivities · 550 unique ligands

UniProt input → EGFR


Resolving identifiers for 'CHEMBL203'...

UniProt: P00533  |  ChEMBL: CHEMBL203

Fetching data from 4 sources in parallel...

Done!  50 structures · 1000 bioactivities · 550 unique ligands

ChEMBL input  → EGFR


## 3. Explore UniProt data

In [4]:
u = report.uniprot

print("Gene name      :", u.gene_name)
print("Protein name   :", u.protein_name)
print("Organism       :", u.organism)
print("UniProt ID     :", u.uniprot_id)
print("ChEMBL target  :", u.chembl_id)
print("Sequence length:", u.sequence_length)
print("\nFunction (first 300 chars):")
print(u.function_description[:300] if u.function_description else "N/A")

Gene name      : EGFR
Protein name   : Epidermal growth factor receptor
Organism       : Homo sapiens
UniProt ID     : P00533
ChEMBL target  : CHEMBL203
Sequence length: 1210

Function (first 300 chars):
Receptor tyrosine kinase binding ligands of the EGF family and activating several signaling cascades to convert extracellular cues into appropriate cellular responses (PubMed:10805725, PubMed:27153536, PubMed:2790960, PubMed:35538033). Known ligands include EGF, TGFA/TGF-alpha, AREG, epigen/EPGN, BT


In [5]:
# Subcellular location
print("Subcellular locations:", u.subcellular_locations)

# Diseases
print("\nAssociated diseases:")
for d in u.disease_associations[:5]:
    print(" -", d)

Subcellular locations: ['Cell membrane', 'Endoplasmic reticulum membrane', 'Golgi apparatus membrane', 'Nucleus membrane', 'Endosome', 'Endosome membrane', 'Nucleus', 'Secreted']

Associated diseases:
 - Lung cancer: A common malignancy affecting tissues of the lung. The most common form of lung cancer is non-small cell lung cancer (NSCLC) that can be divided into 3 major histologic subtypes: squamous cell carcinoma, adenocarcinoma, and large cell lung cancer. NSCLC is often diagnosed at an advanced stage and has a poor prognosis.
 - Neonatal nephrocutaneous inflammatory syndrome: An autosomal recessive disorder characterized by intrauterine growth retardation, premature birth, fragile skin, recurrent skin infections and sepsis, failure to thrive, nephrocalcinosis, and nephromegaly with tubular dysfunction. Some patients have chronic diarrhea, and necrotizing enterocolitis.


In [6]:
# GO terms grouped by category
from collections import defaultdict

go_by_cat = defaultdict(list)
for go in u.go_terms:
    go_by_cat[go.category].append(go.term)

for cat, terms in go_by_cat.items():
    print(f"\nGO — {cat} ({len(terms)} terms):")
    for t in terms[:5]:
        print("  ", t)


GO — cellular_component (32 terms):
   basal plasma membrane
   basolateral plasma membrane
   cell junction
   cell surface
   ciliary basal body

GO — molecular_function (19 terms):
   actin filament binding
   ATP binding
   ATPase binding
   cadherin binding
   chromatin binding

GO — biological_process (53 terms):
   cell morphogenesis
   cell surface receptor signaling pathway
   cell-cell adhesion
   cellular response to amino acid stimulus
   cellular response to epidermal growth factor stimulus


## 4. Explore PDB structures

In [7]:
import pandas as pd

pdb_rows = []
for s in report.pdb_structures:
    pdb_rows.append({
        "PDB ID": s.pdb_id,
        "Method": s.method.value if s.method else "",
        "Resolution (Å)": s.resolution,
        "Deposit Date": s.release_date,
        "Ligands": ", ".join(l.ligand_id for l in s.ligands) if s.ligands else "",
        "Title": (s.title or "")[:60],
    })

pdb_df = pd.DataFrame(pdb_rows)
print(f"{len(pdb_df)} structures")
pdb_df.head(10)

50 structures


,PDB ID,Method,Resolution (Å),Deposit Date,Ligands,Title
0,8A27,X-RAY DIFFRACTION,1.07,2022-06-02T00:00:00+0000,,"EGFR kinase domain in complex with 2-(6,7-dihy..."
1,8A2D,X-RAY DIFFRACTION,1.11,2022-06-03T00:00:00+0000,,EGFR kinase domain (L858R/V948R) in complex wi...
2,5UG9,X-RAY DIFFRACTION,1.33,2017-01-07T00:00:00+0000,8AM,Crystal structure of the EGFR kinase domain (L...
3,5HG8,X-RAY DIFFRACTION,1.42,2016-01-08T00:00:00+0000,634,"EGFR (L858R, T790M, V948R) in complex with N-[..."
4,8A2A,X-RAY DIFFRACTION,1.43,2022-06-03T00:00:00+0000,,"EGFR kinase domain in complex with 2-(6,7-dihy..."
5,5UG8,X-RAY DIFFRACTION,1.46,2017-01-07T00:00:00+0000,8BP,Crystal structure of the EGFR kinase domain (L...
6,3POZ,X-RAY DIFFRACTION,1.50,2010-11-23T00:00:00+0000,,EGFR Kinase domain complexed with tak-285
7,6TFV,X-RAY DIFFRACTION,1.50,2019-11-14T00:00:00+0000,N7Q,Crystal Structure of EGFR T790M/V948R in Compl...
8,6TG0,X-RAY DIFFRACTION,1.50,2019-11-14T00:00:00+0000,N78,Crystal Structure of EGFR T790M/V948R in Compl...
9,3VRP,X-RAY DIFFRACTION,1.52,2012-04-13T00:00:00+0000,,Crystal structure of the tyrosine kinase bindi...


In [8]:
# Method breakdown
print("Method breakdown:")
print(pdb_df["Method"].value_counts().to_string())

# Resolution statistics
res = pdb_df["Resolution (Å)"].dropna()
print(f"\nResolution stats (Å): min={res.min():.2f}  median={res.median():.2f}  max={res.max():.2f}")

Method breakdown:
Method
X-RAY DIFFRACTION    50

Resolution stats (Å): min=1.07  median=1.77  max=1.98


## 5. AlphaFold predicted structure

In [9]:
af = report.alphafold
if af:
    print("AlphaFold UniProt ID:", af.uniprot_id)
    print("Model version       :", af.version)
    print("PDB download URL    :", af.pdb_url)
    print("Mean pLDDT          :", af.mean_plddt)
else:
    print("No AlphaFold entry found.")

AlphaFold UniProt ID: P00533
Model version       : 6
PDB download URL    : https://alphafold.ebi.ac.uk/files/AF-P00533-F1-model_v6.pdb
Mean pLDDT          : None


## 6. Bioactivity data (ChEMBL)

In [10]:
bio_rows = []
for b in report.bioactivities:
    bio_rows.append({
        "Molecule": b.molecule_chembl_id or b.name or "",
        "Activity Type": b.activity_type,
        "Value (nM)": b.value,
        "pChEMBL": b.pchembl_value,
        "Source": b.source,
        "SMILES": (b.smiles or "")[:40],
    })

bio_df = pd.DataFrame(bio_rows)
print(f"{len(bio_df)} bioactivity records")
bio_df.head(10)

1000 bioactivity records


,Molecule,Activity Type,Value (nM),pChEMBL,Source,SMILES
0,CHEMBL176582,IC50,0.010,11.00,ChEMBL,Cn1cnc2cc3ncnc(Nc4cccc(Br)c4)c3cc21
1,CHEMBL4650319,IC50,0.010,11.00,ChEMBL,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn
2,CHEMBL5790648,IC50,0.016,10.80,ChEMBL,COc1cc2ncnc(Nc3cccc(Cl)c3F)c2cc1N1CC2(CN
3,CHEMBL5746224,IC50,0.017,10.77,ChEMBL,CN1CCC2(CC1)CN(c1ccc3ncnc(Nc4cccc(Cl)c4F
4,CHEMBL5270693,IC50,0.020,10.70,ChEMBL,COc1cc(N2CCC(N(C)C)CC2)ccc1Nc1ncc(C(=O)O
5,CHEMBL5997498,IC50,0.020,10.70,ChEMBL,COc1cc2ncnc(Nc3cccc(Cl)c3F)c2cc1N1CC2(CC
6,CHEMBL180022,IC50,0.020,10.70,ChEMBL,CCOc1cc2ncc(C#N)c(Nc3ccc(OCc4ccccn4)c(Cl
7,CHEMBL4537790,IC50,0.020,10.70,ChEMBL,Cc1cc2cc(n1)-c1cnn(C)c1OCCC[C@@H](C)CN1/
8,CHEMBL3353410,IC50,0.020,10.70,ChEMBL,C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)
9,CHEMBL180022,IC50,0.020,10.70,ChEMBL,CCOc1cc2ncc(C#N)c(Nc3ccc(OCc4ccccn4)c(Cl


In [11]:
# Source breakdown
print("Source breakdown:")
print(bio_df["Source"].value_counts().to_string())

# Activity type breakdown
print("\nActivity type breakdown:")
print(bio_df["Activity Type"].value_counts().head(8).to_string())

# pChEMBL distribution
pc = bio_df["pChEMBL"].dropna()
print(f"\npChEMBL stats: min={pc.min():.2f}  median={pc.median():.2f}  max={pc.max():.2f}")
print(f"High-potency (pChEMBL ≥ 9): {(pc >= 9).sum()} records")

Source breakdown:
Source
ChEMBL    1000

Activity type breakdown:
Activity Type
IC50    883
Kd       92
Ki       25

pChEMBL stats: min=9.16  median=9.52  max=11.00
High-potency (pChEMBL ≥ 9): 1000 records


## 7. Unique ligands ranked by potency

In [12]:
lig_rows = []
for l in report.ligand_summary:
    lig_rows.append({
        "Name": l.name or "",
        "ChEMBL ID": l.chembl_id or "",
        "Best pChEMBL": l.best_pchembl,
        "Activity Type": l.best_activity_type,
        "Activity (nM)": l.best_activity_value_nM,
        "# Assays": l.num_assays,
        "Sources": ", ".join(l.sources),
        "SMILES": (l.smiles or "")[:50],
    })

lig_df = pd.DataFrame(lig_rows)
print(f"{len(lig_df)} unique ligands (sorted by pChEMBL descending)")
lig_df.head(20)

550 unique ligands (sorted by pChEMBL descending)


,Name,ChEMBL ID,Best pChEMBL,Activity Type,Activity (nM),# Assays,Sources,SMILES
0,,CHEMBL176582,11.00,IC50,0.010,1,ChEMBL,Cn1cnc2cc3ncnc(Nc4cccc(Br)c4)c3cc21
1,MOBOCERTINIB,CHEMBL4650319,11.00,IC50,0.010,2,ChEMBL,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...
2,,CHEMBL5790648,10.80,IC50,0.016,2,ChEMBL,COc1cc2ncnc(Nc3cccc(Cl)c3F)c2cc1N1CC2(CN(C)C2)...
3,,CHEMBL5746224,10.77,IC50,0.017,2,ChEMBL,CN1CCC2(CC1)CN(c1ccc3ncnc(Nc4cccc(Cl)c4F)c3c1)...
4,,CHEMBL5270693,10.70,IC50,0.020,2,ChEMBL,COc1cc(N2CCC(N(C)C)CC2)ccc1Nc1ncc(C(=O)Oc2cccc...
5,,CHEMBL5997498,10.70,IC50,0.020,2,ChEMBL,COc1cc2ncnc(Nc3cccc(Cl)c3F)c2cc1N1CC2(CCN(C)CC...
6,NERATINIB,CHEMBL180022,10.70,IC50,0.020,11,ChEMBL,CCOc1cc2ncc(C#N)c(Nc3ccc(OCc4ccccn4)c(Cl)c3)c2...
7,,CHEMBL4537790,10.70,IC50,0.020,3,ChEMBL,Cc1cc2cc(n1)-c1cnn(C)c1OCCC[C@@H](C)CN1/C(=N/C...
8,OSIMERTINIB,CHEMBL3353410,10.70,IC50,0.020,22,ChEMBL,C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(O...
9,,CHEMBL6054692,10.68,IC50,0.021,2,ChEMBL,CCCN1CC2(C1)CN(c1cc3c(Nc4cccc(Cl)c4F)ncnc3cc1O...


In [13]:
# Best ligand shortcut
best = report.best_ligand
if best:
    print("Best ligand:")
    print(f"  Name     : {best.name}")
    print(f"  ChEMBL ID: {best.chembl_id}")
    print(f"  pChEMBL  : {best.best_pchembl}")
    print(f"  SMILES   : {best.smiles}")

Best ligand:
  Name     : None
  ChEMBL ID: CHEMBL176582
  pChEMBL  : 11.0
  SMILES   : Cn1cnc2cc3ncnc(Nc4cccc(Br)c4)c3cc21


In [14]:
# Filter ligands programmatically
potent = [l for l in report.ligand_summary if l.best_pchembl and l.best_pchembl >= 9.0]
print(f"Ligands with pChEMBL ≥ 9.0: {len(potent)}")

ic50_only = [l for l in report.ligand_summary if l.best_activity_type == "IC50"]
print(f"Ligands measured by IC50: {len(ic50_only)}")

multi_source = [l for l in report.ligand_summary if len(l.sources) > 1]
print(f"Ligands confirmed in multiple databases: {len(multi_source)}")

Ligands with pChEMBL ≥ 9.0: 550
Ligands measured by IC50: 528
Ligands confirmed in multiple databases: 0


## 8. Protein–protein interactions (STRING DB)

In [15]:
if report.interactions:
    int_rows = []
    for i in report.interactions:
        int_rows.append({
            "Partner": i.gene_b,
            "Score": i.score,
        })
    int_df = pd.DataFrame(int_rows).sort_values("Score", ascending=False)
    print(f"{len(int_df)} protein interactions")
    print(int_df.to_string(index=False))
else:
    print("No interaction data.")

30 protein interactions
 Partner  Score
  PTPN11  0.999
   HBEGF  0.999
    NRG1  0.999
    TGFA  0.999
   ERBB2  0.999
     DCN  0.999
    SHC1  0.999
    SOS1  0.999
  PIK3CA  0.999
   ERBB3  0.999
    GRB2  0.999
    EPGN  0.999
    CAV1  0.999
    MUC1  0.999
     BTC  0.999
HSP90AA1  0.999
    GAB1  0.999
     SRC  0.999
    EREG  0.999
    AREG  0.999
     CBL  0.999
    PTK2  0.999
    CDH1  0.999
     EGF  0.999
  PIK3R1  0.998
  CTNNB1  0.998
    CD44  0.998
    SOS2  0.998
  STAT5A  0.998
  PDGFRB  0.998


## 9. Export reports

In [19]:
from targetrecon.core import save_html, save_json, save_sdf
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)

# HTML — interactive self-contained report
html_path = save_html(report, out / "EGFR_report.html")
print("HTML report  →", html_path)

# JSON — full machine-readable report
json_path = save_json(report, out / "EGFR_report.json")
print("JSON report  →", json_path)

# SDF — top 20 ligands with 3D conformers (RDKit MMFF)
sdf_path = save_sdf(report, out / "EGFR_top20_ligands.sdf", top_n=20)
print("SDF ligands  →", sdf_path)

HTML report  → outputs/EGFR_report.html
JSON report  → outputs/EGFR_report.json
SDF ligands  → outputs/EGFR_top20_ligands.sdf


In [20]:
# SDF with filters — only IC50, pChEMBL ≥ 8, top 50
sdf_filtered = save_sdf(
    report,
    out / "EGFR_IC50_filtered.sdf",
    top_n=50,
    min_pchembl=8.0,
    activity_type="IC50",
)
print("Filtered SDF →", sdf_filtered)

Filtered SDF → outputs/EGFR_IC50_filtered.sdf


## 10. Batch processing — compare targets

Fetch multiple targets concurrently using `asyncio.gather()` via `recon_async()`.

In [16]:
import asyncio

panel = ["EGFR", "BRAF", "CDK2", "ABL1"]

panel_reports = await asyncio.gather(*[
    targetrecon.recon_async(t, verbose=False) for t in panel
])

summary = []
for t, r in zip(panel, panel_reports):
    if r.uniprot is None:
        continue
    summary.append({
        "Target": t,
        "UniProt": r.uniprot.uniprot_id,
        "PDB Structures": r.num_pdb_structures,
        "Bioactivities": r.num_bioactivities,
        "Unique Ligands": r.num_unique_ligands,
        "Best pChEMBL": r.best_ligand.best_pchembl if r.best_ligand else None,
        "Best Ligand": r.best_ligand.name or r.best_ligand.chembl_id if r.best_ligand else None,
    })

pd.DataFrame(summary)

,Target,UniProt,PDB Structures,Bioactivities,Unique Ligands,Best pChEMBL,Best Ligand
0,EGFR,P00533,50,1000,550,11.00,CHEMBL176582
1,BRAF,P15056,50,1000,572,10.70,CHEMBL5885861
2,CDK2,P24941,50,1000,711,10.05,EBVACICLIB
3,ABL1,P00519,50,1000,622,10.82,CHEMBL2347725


In [21]:
# Export all panel reports as HTML
for t, r in zip(panel, panel_reports):
    if r.uniprot:
        p = save_html(r, out / f"{t}_report.html")
        print(f"  {t} → {p}")

  EGFR → outputs/EGFR_report.html
  BRAF → outputs/BRAF_report.html
  CDK2 → outputs/CDK2_report.html
  ABL1 → outputs/ABL1_report.html


## 11. Work with raw JSON

The `TargetReport` is a Pydantic model — serialize/deserialize freely.

In [22]:
# Serialize to dict
data = report.model_dump()
print("Top-level keys:", list(data.keys()))

# Serialize to JSON string
json_str = report.model_dump_json(indent=2)
print(f"\nJSON size: {len(json_str):,} characters")

# Load back from JSON file
from targetrecon.models import TargetReport
with open(out / "EGFR_report.json") as f:
    loaded = TargetReport.model_validate_json(f.read())
print(f"\nRound-trip: {loaded.uniprot.gene_name} | {loaded.num_unique_ligands} ligands")

Top-level keys: ['query', 'generated_at', 'targetrecon_version', 'uniprot', 'pdb_structures', 'alphafold', 'bioactivities', 'ligand_summary', 'num_pdb_structures', 'num_bioactivities', 'num_unique_ligands', 'best_ligand', 'interactions', 'ai_analysis']

JSON size: 531,404 characters

Round-trip: EGFR | 550 ligands


## 12. Quick visualization (optional — requires matplotlib)

In [ ]:
try:
    import matplotlib.pyplot as plt
    from collections import Counter

    pchembl_vals = [b.pchembl_value for b in report.bioactivities if b.pchembl_value]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # pChEMBL histogram
    axes[0].hist(pchembl_vals, bins=30, color="#58a6ff", edgecolor="white", linewidth=0.5)
    axes[0].axvline(7, color="#f85149", linestyle="--", label="pChEMBL = 7 (100 nM)")
    axes[0].axvline(9, color="#3fb950", linestyle="--", label="pChEMBL = 9 (1 nM)")
    axes[0].set_xlabel("pChEMBL value")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"EGFR — pChEMBL distribution (n={len(pchembl_vals)})")
    axes[0].legend()

    # Activity type bar chart
    atype_counts = Counter(b.activity_type for b in report.bioactivities if b.activity_type)
    top_types = dict(atype_counts.most_common(6))
    axes[1].bar(top_types.keys(), top_types.values(), color="#bc8cff", edgecolor="white")
    axes[1].set_xlabel("Activity type")
    axes[1].set_ylabel("Count")
    axes[1].set_title("EGFR — Bioactivity type breakdown")

    plt.tight_layout()
    plt.savefig(out / "EGFR_charts.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved →", out / "EGFR_charts.png")

except ImportError:
    print("matplotlib not installed — pip install matplotlib to enable visualizations")